In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
import re
import nltk
nltk.download('punkt')

# Device selection: use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == 'cuda':
    # clear cache and enable cuDNN benchmark for potential speedups
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True

c:\Users\Bobby\Desktop\MasterThesis\Multilingual_Characterization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Bobby\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
DATA_ROOT = "../data"
TRAIN_DATA_PARENT = os.path.join(DATA_ROOT, "target_4_December_release")
VAL_DATA_PARENT = os.path.join(DATA_ROOT, "cleaned_dev_10_january_2025")
TEST_DATA_PARENT = os.path.join(DATA_ROOT, "testdata_ST12")
TAXONOMY_FILE = os.path.join(DATA_ROOT, "taxonomy.json")

# Baseline-specific output paths (isolated from advanced model outputs)
BASELINE_CHECKPOINT_DIR = "../checkpoints/baseline_classifier"
BASELINE_PREDICTIONS_DIR = "../predictions/baseline"
BASELINE_DIAGRAMS_DIR = "../diagrams/baseline"

os.makedirs(BASELINE_PREDICTIONS_DIR, exist_ok=True)
os.makedirs(BASELINE_DIAGRAMS_DIR, exist_ok=True)

# Use validation set as test set (no separate test labels available)
TEST_DATA_PARENT = VAL_DATA_PARENT

In [3]:
# === Detect all available language folders ===
available_languages = [d for d in os.listdir(TRAIN_DATA_PARENT) if os.path.isdir(os.path.join(TRAIN_DATA_PARENT, d))]
print("Detected languages:", available_languages)

with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

label_data = []

for category in taxonomy:
    for subtype in category["subtypes"]:
        label_data.append({
            "main_category": category["name"],
            "subtype": subtype["name"],
            "description": subtype["description"],
            "example": subtype["example"]
        })

df = pd.DataFrame(label_data)
df.head()

Detected languages: ['BG', 'EN', 'HI', 'PT', 'RU']


,main_category,subtype,description,example
0,Protagonist,Guardian,A person who protects or defends something; or...,Police officers protecting citizens during a c...
1,Protagonist,Martyr,A person who is killed or who suffers greatly ...,Civil rights leaders like Martin Luther King J...
2,Protagonist,Peacemaker,"A person, group, or nation that tries to make ...",Nelson Mandela's efforts to reconcile South Af...
3,Protagonist,Rebel,A person who fights against the government of ...,Leaders of independence movements like Mahatma...
4,Protagonist,Underdog,"A person, team, or entity that is expected to ...",Grassroots political candidates overcoming wel...


In [4]:
main_categories = [cat["name"] for cat in taxonomy]
label2id = {label: i for i, label in enumerate(main_categories)}
id2label = {i: label for label, i in label2id.items()}

In [5]:
# === Dataset preparation ===
# === Dataset loader ===
def load_annotations(annotation_path, docs_root, labeled=True):
    data = []
    with open(annotation_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if labeled:
                # if len(parts) < 5:
                #     continue
                doc_id, mention, start, end, *labels = parts
                label = labels[0]
            else:
                # if len(parts) < 4:
                #     continue
                doc_id, mention, start, end = parts
                label = None

            start, end = int(start), int(end)
            text_path = os.path.join(docs_root, doc_id)
            # if not os.path.exists(text_path):
            #     continue

            with open(text_path, "r", encoding="utf-8") as doc_file:
                text = doc_file.read()

            entry = {
                "doc_id": doc_id,
                "text": text,
                "mention": mention,
                "start": start,
                "end": end,
            }
            if labeled:
                entry["label"] = label
            data.append(entry)
    return pd.DataFrame(data)

import nltk
nltk.download("punkt")

# --------------------------------------------------------------------
# Utility: given text, produce (span_start, span_end, substring)
# for paragraphs or sentences, maintaining global offsets.
# --------------------------------------------------------------------
def split_paragraphs_with_offsets(text):
    parts = []
    offset = 0
    for raw in text.split("\n"):
        p = raw.strip()
        if not p:
            offset += len(raw) + 1  # still move offset
            continue
        start = text.index(raw, offset)
        end = start + len(raw) - 1
        parts.append((start, end, raw))
        offset = end + 1
    return parts


def split_sentences_with_offsets(text):
    parts = []
    sentences = nltk.sent_tokenize(text)

    search_offset = 0
    for s in sentences:
        idx = text.find(s, search_offset)
        if idx == -1:
            idx = text.index(s)  # fallback
        parts.append((idx, idx + len(s), s))
        search_offset = idx + len(s)
    return parts


# --------------------------------------------------------------------
# Main expander: document → paragraphs or sentences
# This enforces correct mention span logic.
# --------------------------------------------------------------------
def expand_to_smaller_units(df, mode):
    new_rows = []

    for _, row in df.iterrows():
        text = row["text"]
        mention = row["mention"]
        orig_start = row["start"]
        orig_end = row["end"]

        # ----------------------------------------------------------------
        # choose unit splitter
        # ----------------------------------------------------------------
        if mode == "paragraph":
            units = split_paragraphs_with_offsets(text)
        elif mode == "sentence":
            units = split_sentences_with_offsets(text)
        else:
            raise ValueError("Unsupported mode.")

        # ----------------------------------------------------------------
        # Only keep the chunk that actually contains the original mention
        # ----------------------------------------------------------------
        for unit_start, unit_end, unit_text in units:

            # Does this unit include the absolute mention span?
            if not (unit_start <= orig_start < unit_end):
                continue

            # ----------------------------------------------------------------
            # Recalculate local start/end inside this chunk
            # ----------------------------------------------------------------
            mention_length = orig_end - orig_start + 1

            local_start = orig_start - unit_start
            local_end   = local_start + mention_length - 1


            # Additional safety check:
            if unit_text[local_start:local_end] != mention:
                # fallback: search within the chunk for exact location
                # This handles rare tokenization boundary issues
                idx = unit_text.find(mention)
                if idx == -1:
                    continue
                local_start = idx
                local_end = idx + len(mention) - 1

            # Build new row
            new_row = row.copy()
            new_row["text"] = unit_text

            # local offsets for the model
            new_row["start"] = local_start
            new_row["end"] = local_end

            # store ORIGINAL global offsets (critical!)
            new_row["orig_start"] = orig_start
            new_row["orig_end"] = orig_end

            new_rows.append(new_row)


    return pd.DataFrame(new_rows)


# === Combine data from all languages ===
def load_multilingual_data_by_mode(mode="document", labeled=True):

    assert mode in ["document", "paragraph", "sentence"]

    all_train, all_val, all_test = [], [], []

    for lang in available_languages:
        print(f"\n📘 Loading {mode}-level data for language: {lang}")

        train_root = os.path.join(TRAIN_DATA_PARENT, lang, "raw-documents")
        train_ann  = os.path.join(TRAIN_DATA_PARENT, lang, "subtask-1-annotations.txt")

        val_root = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-documents")
        val_ann  = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-annotations.txt")

        test_root = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-documents")
        test_ann  = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-annotations.txt")

        if not (os.path.exists(train_ann) and os.path.exists(val_ann)):
            print(f"⚠️ Skipping {lang}: missing annotation files")
            continue

        # ======================
        # Load document-level data
        # ======================
        train_df_doc = load_annotations(train_ann, train_root, labeled=labeled)
        val_df_doc   = load_annotations(val_ann, val_root, labeled=labeled)
        test_df_doc  = load_annotations(test_ann, test_root, labeled=labeled) if os.path.exists(test_ann) else pd.DataFrame()

        # ======================
        # Transform based on mode
        # ======================
        if mode == "document":
            train_df, val_df, test_df = train_df_doc, val_df_doc, test_df_doc

        else:
            train_df = expand_to_smaller_units(train_df_doc, mode)
            val_df   = expand_to_smaller_units(val_df_doc, mode)
            test_df  = expand_to_smaller_units(test_df_doc, mode) if not test_df_doc.empty else pd.DataFrame()

        # append
        all_train.append(train_df)
        all_val.append(val_df)
        all_test.append(test_df)

    return (
        pd.concat(all_train, ignore_index=True),
        pd.concat(all_val, ignore_index=True),
        pd.concat(all_test, ignore_index=True)
    )

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Bobby\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
# Load the data
train_df_full, val_df, test_df = load_multilingual_data_by_mode(mode="paragraph", labeled=True)

print(f"\n✅ Loaded dataset sizes: Train={len(train_df_full)}, Val={len(val_df)}, Test={len(test_df)}")

# Split the training data into new train and validation sets
train_df, new_val_df = train_test_split(train_df_full, test_size=0.2, random_state=42, stratify=train_df_full['label'])

# Use new validation set and keep using original validation set as test set
val_df = new_val_df
print(f"\n✅ Final dataset sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Print class distribution
print("\n=== Label distribution ===")
print("Training set:")
print(train_df['label'].value_counts().to_string())
print("\nValidation set:")
print(val_df['label'].value_counts().to_string())
print("\nTest set:")
print(test_df['label'].value_counts().to_string())

train_df.head()


📘 Loading paragraph-level data for language: BG

📘 Loading paragraph-level data for language: EN

📘 Loading paragraph-level data for language: HI

📘 Loading paragraph-level data for language: PT

📘 Loading paragraph-level data for language: RU

✅ Loaded dataset sizes: Train=5606, Val=604, Test=604

✅ Final dataset sizes: Train=4484, Val=1122, Test=604

=== Label distribution ===
Training set:
label
Antagonist     2108
Protagonist    1491
Innocent        885

Validation set:
label
Antagonist     528
Protagonist    373
Innocent       221

Test set:
label
Antagonist     266
Protagonist    191
Innocent       147


,doc_id,text,mention,start,end,label,orig_start,orig_end
1865,HI_307.txt,मॉस्को: उत्तर कोरिया ने कहा कि वह रूस के साथ उ...,रूस,34,36,Protagonist,147,149
4315,PT_64.txt,A Procuradoria-Geral da Ucrânia afirma que os ...,Procuradoria-Geral da Ucrânia,2,30,Protagonist,1487,1515
769,EN_UA_026036.txt,KHARKIV - Russia pounded over 30 villages and...,Russia,11,16,Antagonist,77,82
3875,PT_27.txt,"""Repito mais uma vez sobre a existência do dec...",Zelensky,54,61,Antagonist,2289,2296
5080,RU-URW-1249.txt,"После ракетного удара по Киеву, в результате к...",Владимира Зеленского,362,381,Antagonist,362,381


In [7]:
# === Dataset class ===
class EntityFramingDataset(Dataset):
    def __init__(self, df, tokenizer, label2id, max_len=256, labeled=True):
        self.df = df
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len
        self.labeled = labeled

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text"]
        mention = row["mention"]
        marked_text = text.replace(mention, f"[ENTITY] {mention} [/ENTITY]")

        inputs = self.tokenizer(
            marked_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in inputs.items()}

        if self.labeled:
            label = self.label2id[row["label"]]
            item["labels"] = torch.tensor(label, dtype=torch.long)

        return item


In [8]:
# === Metrics ===
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="micro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [9]:
# === Tokenizer and Model ===
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = EntityFramingDataset(train_df, tokenizer, label2id, labeled=True)
val_dataset = EntityFramingDataset(val_df, tokenizer, label2id, labeled=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Freeze all layers except the last transformer block, pooler, and classifier
for name, param in model.named_parameters():
    if not any(x in name for x in ['pooler', 'classifier', 'encoder.layer.11']):
        param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Percentage of trainable parameters: {100 * trainable_params / total_params:.2f}%')

model.to(device)

# === Training ===
training_args = TrainingArguments(
    output_dir=BASELINE_CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

# === Evaluation ===
results = trainer.evaluate()
print("\n=== Validation Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 278,045,955
Trainable parameters: 7,680,771
Percentage of trainable parameters: 2.76%


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.933700,0.829649,0.636364,0.636364,0.636364,0.636364
2,0.908500,0.769006,0.670232,0.670232,0.670232,0.670232
3,0.962500,0.750744,0.686275,0.686275,0.686275,0.686275
4,0.702500,0.755771,0.682709,0.682709,0.682709,0.682709
5,0.711700,0.743293,0.679144,0.679144,0.679144,0.679144



=== Validation Results ===
eval_loss: 0.7507
eval_accuracy: 0.6863
eval_precision: 0.6863
eval_recall: 0.6863
eval_f1: 0.6863
eval_runtime: 20.2659
eval_samples_per_second: 55.3640
eval_steps_per_second: 6.9570
epoch: 5.0000


In [ ]:
# === Generate and save predictions on test/val set ===
print("=== Generating predictions ===")

test_dataset = EntityFramingDataset(test_df, tokenizer, label2id, labeled=True)
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)
pred_labels = [id2label[i] for i in preds]

test_df = test_df.copy()
test_df["predicted_label"] = pred_labels

# Detect language from doc_id
# Doc IDs can be: "BG_123.txt", "EN_UA_026036.txt", "A9_BG_3970.txt",
# "A6_CC_BG_123.txt", "RU-URW-1249.txt", "HI_307.txt", etc.
def detect_language(doc_id):
    for lang in available_languages:
        # Match: starts with "LANG_" or "LANG-", or contains "_LANG_" / "_LANG-" / "-LANG-" / "-LANG_"
        if (doc_id.startswith(f"{lang}_") or doc_id.startswith(f"{lang}-")
                or f"_{lang}_" in doc_id or f"_{lang}-" in doc_id
                or f"-{lang}_" in doc_id or f"-{lang}-" in doc_id):
            return lang
    return "Unknown"

test_df["language"] = test_df["doc_id"].apply(detect_language)

# Verify language detection
lang_counts = test_df["language"].value_counts()
print(f"\nLanguage distribution:")
for lang, count in lang_counts.items():
    print(f"  {lang}: {count}")
if "Unknown" in lang_counts.index:
    unknown_ids = test_df[test_df["language"] == "Unknown"]["doc_id"].unique()
    print(f"\n⚠️ Unknown doc_ids: {unknown_ids[:5]}")

# Save predictions CSV
pred_path = os.path.join(BASELINE_PREDICTIONS_DIR, "baseline_predictions.csv")
test_df[["doc_id", "mention", "start", "end", "language", "label", "predicted_label"]].to_csv(
    pred_path, index=False
)
print(f"\nPredictions saved to: {pred_path}")
print(f"Total predictions: {len(test_df)}")
test_df.head()

In [11]:
# =====================================
# C1. Training Loss Curve
# =====================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import json
import glob

# Find the last checkpoint with trainer_state.json
ckpt_dirs = sorted(glob.glob(os.path.join(BASELINE_CHECKPOINT_DIR, "checkpoint-*")))
trainer_state_path = None
for d in reversed(ckpt_dirs):
    p = os.path.join(d, "trainer_state.json")
    if os.path.exists(p):
        trainer_state_path = p
        break

# Fallback: use trainer.state.log_history directly if available
if trainer_state_path:
    with open(trainer_state_path, "r") as f:
        state = json.load(f)
    log_history = state["log_history"]
else:
    log_history = trainer.state.log_history

train_steps = [e["step"] for e in log_history if "loss" in e]
train_loss = [e["loss"] for e in log_history if "loss" in e]
eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
eval_loss = [e["eval_loss"] for e in log_history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_loss, label="Training Loss", alpha=0.7, linewidth=1.5)
if eval_loss:
    ax.plot(eval_steps, eval_loss, label="Validation Loss", marker='o', linewidth=2)
ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Baseline Model: Training Loss Curve", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c1_baseline_training_loss.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c1_baseline_training_loss.png")

Saved: c1_baseline_training_loss.png


C:\Users\Bobby\AppData\Local\Temp\ipykernel_17064\1532996162.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# =====================================
# C2. Confusion Matrix
# =====================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = test_df["label"].tolist()
y_pred = test_df["predicted_label"].tolist()
labels_order = ["Protagonist", "Antagonist", "Innocent"]

cm = confusion_matrix(y_true, y_pred, labels=labels_order)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Baseline Model: Confusion Matrix", fontsize=14, fontweight='bold')
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c2_baseline_confusion_matrix.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c2_baseline_confusion_matrix.png")

Saved: c2_baseline_confusion_matrix.png


C:\Users\Bobby\AppData\Local\Temp\ipykernel_17064\3953635376.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# =====================================
# C3. Per-Language F1
# =====================================

# Safety check: filter out any remaining "Unknown" language entries
test_df_lang = test_df[test_df["language"] != "Unknown"].copy()
if len(test_df_lang) < len(test_df):
    print(f"⚠️ Filtered out {len(test_df) - len(test_df_lang)} samples with unknown language")

lang_scores = []
for lang in sorted(test_df_lang["language"].unique()):
    mask = test_df_lang["language"] == lang
    yt = [label2id[l] for l in test_df_lang.loc[mask, "label"]]
    yp = [label2id[l] for l in test_df_lang.loc[mask, "predicted_label"]]
    acc = accuracy_score(yt, yp)
    _, _, f1, _ = precision_recall_fscore_support(yt, yp, average="weighted", zero_division=0)
    lang_scores.append({"Language": lang, "Accuracy": acc, "Weighted F1": f1, "Samples": mask.sum()})

lang_df = pd.DataFrame(lang_scores).sort_values("Weighted F1", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(lang_df["Language"], lang_df["Weighted F1"], color="#4C72B0", edgecolor="white")
for bar, val, n in zip(bars, lang_df["Weighted F1"], lang_df["Samples"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.1%} (n={n})", va='center', fontsize=11)
ax.set_xlabel("Weighted F1", fontsize=12)
ax.set_title("Baseline Model: Weighted F1 per Language", fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.15)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c3_baseline_per_language_f1.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c3_baseline_per_language_f1.png")
print(lang_df.to_string(index=False))

In [14]:
# =====================================
# C4. Per-Class Precision / Recall / F1
# =====================================

y_true_ids = [label2id[l] for l in test_df["label"]]
y_pred_ids = [label2id[l] for l in test_df["predicted_label"]]

class_prec, class_rec, class_f1, class_sup = precision_recall_fscore_support(
    y_true_ids, y_pred_ids, average=None, zero_division=0
)

classes = list(label2id.keys())
x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, class_prec, width, label="Precision", color="#4C72B0")
ax.bar(x,         class_rec,  width, label="Recall",    color="#DD8452")
ax.bar(x + width, class_f1,   width, label="F1",        color="#55A868")

for i in range(len(classes)):
    ax.text(x[i] - width, class_prec[i] + 0.02, f"{class_prec[i]:.2f}", ha='center', fontsize=9)
    ax.text(x[i],         class_rec[i]  + 0.02, f"{class_rec[i]:.2f}",  ha='center', fontsize=9)
    ax.text(x[i] + width, class_f1[i]   + 0.02, f"{class_f1[i]:.2f}",  ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Baseline Model: Per-Class Metrics", fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(BASELINE_DIAGRAMS_DIR, "c4_baseline_per_class_metrics.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: c4_baseline_per_class_metrics.png")

# Print summary
overall_acc = accuracy_score(y_true_ids, y_pred_ids)
_, _, overall_f1, _ = precision_recall_fscore_support(y_true_ids, y_pred_ids, average="weighted", zero_division=0)
print(f"\nOverall Accuracy: {overall_acc:.4f}")
print(f"Overall Weighted F1: {overall_f1:.4f}")
print(f"\nPer-class breakdown:")
for i, cls in enumerate(classes):
    print(f"  {cls}: P={class_prec[i]:.3f}  R={class_rec[i]:.3f}  F1={class_f1[i]:.3f}  Support={class_sup[i]}")

Saved: c4_baseline_per_class_metrics.png

Overall Accuracy: 0.6871
Overall Weighted F1: 0.6822

Per-class breakdown:
  Protagonist: P=0.695  R=0.644  F1=0.668  Support=191
  Antagonist: P=0.641  R=0.820  F1=0.719  Support=266
  Innocent: P=0.851  R=0.503  F1=0.632  Support=147


C:\Users\Bobby\AppData\Local\Temp\ipykernel_17064\222032789.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
